In [ ]:
# Notebook: Crop Stage Dataset Builder (GEE + NASA POWER)
# Paste into Google Colab. Run cell-by-cell.
# Requirements:
# - Google Earth Engine account (optional but recommended for real NDVI)
# - Internet access (Colab has it)
# - This notebook can fall back to synthetic NDVI if GEE auth is not done.

# 0) Install / Imports
%pip install earthengine-api==0.1.352 requests retrying pandas geopandas tqdm

import os
import time
import math
import json
import requests
import datetime
from datetime import date, timedelta
import pandas as pd
import numpy as np
from tqdm import tqdm

# Optional Earth Engine import - will require authentication if used
try:
    import ee
    ee_available = True
except Exception:
    ee_available = False

print("earthengine available:", ee_available)

# ---------------------------------------------------
# 1) CONFIG: crops, default durations (days), default sowing (approx)
# You can edit these defaults as needed.
# Durations are typical full-season durations (approx). Sowing date set as a representative 'optimal' start.
# These are rough defaults for India across typical agro-ecologies.
CROPS = {
    "rice":       {"duration_days": 120, "default_sowing_month_day": (6,15)},   # Kharif rice: mid-June
    "wheat":      {"duration_days": 120, "default_sowing_month_day": (11,15)},  # Rabi wheat: mid-Nov
    "maize":      {"duration_days": 110, "default_sowing_month_day": (6,15)},   # Kharif maize
    "soybean":    {"duration_days": 110, "default_sowing_month_day": (7,1)},    # Kharif soybean
    "cotton":     {"duration_days": 150, "default_sowing_month_day": (6,15)},   # Kharif cotton
    "groundnut":  {"duration_days": 110, "default_sowing_month_day": (6,15)},   # Kharif groundnut
    "sorghum":    {"duration_days": 120, "default_sowing_month_day": (6,15)},   # Jowar
    "bajra":      {"duration_days": 100, "default_sowing_month_day": (6,15)},   # Pearl millet
    "pigeonpea":  {"duration_days": 150, "default_sowing_month_day": (6,15)},   # Tur (longer)
    "sugarcane":  {"duration_days": 360, "default_sowing_month_day": (2,1)},    # sugarcane planting (approx)
}

# ---------------------------------------------------
# 2) Representative major farming districts (one point per major ag state/region)
# These are representative coordinates (lat, lon) for major agricultural areas across India.
# You may replace with your own list or upload a CSV with precise district centroids.
DISTRICT_POINTS = [
    {"name":"Ludhiana_PB","state":"Punjab","lat":30.9,"lon":75.85},
    {"name":"Hisar_HR","state":"Haryana","lat":29.15,"lon":75.72},
    {"name":"Kanpur_UP","state":"Uttar Pradesh","lat":26.45,"lon":80.34},
    {"name":"Patna_BR","state":"Bihar","lat":25.61,"lon":85.13},
    {"name":"Indore_MP","state":"Madhya Pradesh","lat":22.72,"lon":75.86},
    {"name":"Nagpur_MH","state":"Maharashtra","lat":21.15,"lon":79.09},
    {"name":"Surat_GJ","state":"Gujarat","lat":21.17,"lon":72.83},
    {"name":"Kolkata_WB","state":"West Bengal","lat":22.57,"lon":88.36},
    {"name":"Cuttack_OR","state":"Odisha","lat":20.46,"lon":85.88},
    {"name":"Raipur_CG","state":"Chhattisgarh","lat":21.25,"lon":81.63},
    {"name":"Hyderabad_TG","state":"Telangana","lat":17.38,"lon":78.48},
    {"name":"Vijayawada_AP","state":"Andhra_Pradesh","lat":16.52,"lon":80.63},
    {"name":"Dharwad_KA","state":"Karnataka","lat":15.47,"lon":75.01},
    {"name":"Coimbatore_TN","state":"Tamil_Nadu","lat":11.01,"lon":76.96},
    {"name":"Thrissur_KL","state":"Kerala","lat":10.52,"lon":76.21},
    {"name":"Nagaon_AS","state":"Assam","lat":26.35,"lon":92.68},
    {"name":"Jaipur_RJ","state":"Rajasthan","lat":26.91,"lon":75.79},
    {"name":"Srinagar_JK","state":"Jammu_Kashmir","lat":34.08,"lon":74.79},
    {"name":"Dehradun_UT","state":"Uttarakhand","lat":30.32,"lon":78.03},
    {"name":"Pune_MH","state":"Maharashtra","lat":18.52,"lon":73.85}
]
# You can add more points or replace these.

# ---------------------------------------------------
# 3) Utilities: NASA POWER fetcher (daily)
def fetch_nasa_power_daily(lat, lon, start_date, end_date, parameters=None):
    """
    Fetch daily NASA POWER data between start_date and end_date (YYYY-MM-DD).
    Returns pandas DataFrame with date, T2M (temp), PRECTOT (precip), RH2m (relative humidity) if available.
    """
    if parameters is None:
        parameters = ["T2M_MAX","T2M_MIN","T2M","PRECTOT","RH2M"]
    base = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "start": start_date.replace("-",""),
        "end": end_date.replace("-",""),
        "latitude": lat,
        "longitude": lon,
        "parameters": ",".join(parameters),
        "community": "AG",
        "format": "JSON"
    }
    try:
        r = requests.get(base, params=params, timeout=30)
        r.raise_for_status()
        js = r.json()
        data = js.get("properties", {}).get("parameter", {})
        if not data:
            return pd.DataFrame()
        records = []
        # data is like {'T2M':[...], 'PRECTOT':[...] } keyed by param, with date keys
        dates = list(data[ list(data.keys())[0] ].keys())
        for d in dates:
            rec = {"date": pd.to_datetime(d)}
            for p in data.keys():
                rec[p] = data[p][d]
            records.append(rec)
        df = pd.DataFrame(records)
        df = df.sort_values("date")
        return df
    except Exception as e:
        print("NASA POWER fetch error:", e)
        return pd.DataFrame()

# ---------------------------------------------------
# 4) Earth Engine helper: NDVI time-series for a single point
def init_ee_auth():
    """
    Initialize earth engine - runs interactive auth in Colab.
    """
    global ee_available
    try:
        import ee
        ee.Initialize()
        ee_available = True
        print("EE initialized (already authenticated).")
        return True
    except Exception as e:
        print("Attempting interactive EE auth...", e)
    # Try to authenticate (only works interactively)
    try:
        from google.colab import auth
        auth.authenticate_user()
        import ee
        ee.Authenticate()
        ee.Initialize()
        ee_available = True
        print("EE authenticated and initialized.")
        return True
    except Exception as e:
        print("EE auth failed or running outside Colab. Error:", e)
        ee_available = False
        return False

def get_ndvi_timeseries_ee(lat, lon, start_date, end_date, buffer_m=250):
    """
    Returns a pandas DataFrame with date and NDVI for the point buffered area.
    Requires Earth Engine initialized.
    """
    if not ee_available:
        print("Earth Engine not available. Returning empty DataFrame.")
        return pd.DataFrame()
    try:
        import ee
        pt = ee.Geometry.Point([float(lon), float(lat)]).buffer(buffer_m)
        # Sentinel-2 Surface Reflectance
        collection = (ee.ImageCollection("COPERNICUS/S2_SR")
                      .filterBounds(pt)
                      .filterDate(start_date, end_date)
                      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
                     )
        def add_ndvi(img):
            nd = img.normalizedDifference(['B8','B4']).rename('NDVI')
            return img.addBands(nd)
        coll_ndvi = collection.map(add_ndvi).select('NDVI')
        # Sample mean NDVI for each image and get date
        def to_feature(img):
            mean = img.reduceRegion(ee.Reducer.mean(), pt, 30).get('NDVI')
            date = img.date().format('YYYY-MM-dd')
            return ee.Feature(None, {'date': date, 'NDVI': mean})
        feats = coll_ndvi.map(to_feature).filter(ee.Filter.notNull(['NDVI']))
        ft_list = feats.getInfo()['features'] if feats.size().getInfo() > 0 else []
        rows = []
        for f in ft_list:
            props = f['properties']
            rows.append({'date': pd.to_datetime(props['date']), 'NDVI': props['NDVI']})
        if not rows:
            return pd.DataFrame()
        df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
        return df
    except Exception as e:
        print("EE NDVI fetch error:", e)
        return pd.DataFrame()

# ---------------------------------------------------
# 5) Synthetic NDVI generator (fallback)
def generate_synthetic_ndvi(start_date, end_date, peak_day_frac=0.45, noise=0.03):
    """
    Generate a synthetic NDVI time series between start_date and end_date.
    NDVI increases, peaks, then decreases - shaped like a gamma/sine curve.
    """
    sd = pd.to_datetime(start_date)
    ed = pd.to_datetime(end_date)
    days = (ed - sd).days + 1
    x = np.arange(days)
    # smooth rise and fall: use a scaled sine for simplicity
    ndvi = 0.2 + 0.65 * np.sin(np.clip(np.pi * x / (days * 1.0 / peak_day_frac), 0, np.pi))
    ndvi = ndvi + np.random.normal(0, noise, size=ndvi.shape)
    ndvi = np.clip(ndvi, 0.02, 0.95)
    dates = [sd + timedelta(days=int(i)) for i in x]
    df = pd.DataFrame({"date": dates, "NDVI": ndvi})
    return df

# ---------------------------------------------------
# 6) Stage labeling function (simple DAS-based labels)
def label_stage_by_days(days_since_sowing, crop_duration):
    """
    Basic fixed-window labeler. Windows are approximate and can be customized per crop.
    Returns one of: Germination, Vegetative, Flowering, GrainFill, Maturity/Harvest
    """
    d = days_since_sowing
    total = crop_duration
    # Define windows proportionally (approximate)
    if d < 0:
        return "PreSowing"
    if d <= max(14, int(0.12 * total)):
        return "Germination"
    elif d <= max(45, int(0.4 * total)):
        return "Vegetative"
    elif d <= max(75, int(0.7 * total)):
        return "Flowering"
    elif d <= total:
        return "GrainFill"
    else:
        return "Harvest"

# ---------------------------------------------------
# 7) Main pipeline: loop crops x districts, fetch NDVI + Weather, create labeled csv
OUTPUT_DIR = "crop_stage_datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ask user if they want to initialize EE (interactive in Colab)
print("\nEarth Engine available:", ee_available)
if not ee_available:
    print("If you want to use real Sentinel-2 NDVI, run the init_ee_auth() cell below to authenticate Earth Engine.")
else:
    print("Earth Engine is available. You can fetch real NDVI.")

# (Optional) If you want to auto-init EE here uncomment:
# init_ee_auth()

# Processing parameters
buffer_m = 250  # buffer around point to compute mean NDVI
use_ee_if_possible = True  # if False, always use synthetic NDVI

# Iterate
summary_rows = []
for crop, cinfo in CROPS.items():
    dur = cinfo["duration_days"]
    default_month, default_day = cinfo["default_sowing_month_day"]
    print(f"\n== Processing crop: {crop} (duration {dur} days) ==")
    for pt in tqdm(DISTRICT_POINTS):
        name = pt["name"]
        lat = pt["lat"]
        lon = pt["lon"]

        # Compute a representative sowing date: use current year with default month/day
        year = datetime.date.today().year
        sowing_dt = datetime.date(year, default_month, default_day)
        # If sowing date in future for this year, use previous year (so data exists)
        if sowing_dt > datetime.date.today():
            sowing_dt = datetime.date(year-1, default_month, default_day)
        start_date = sowing_dt.strftime("%Y-%m-%d")
        end_date = (sowing_dt + timedelta(days=dur)).strftime("%Y-%m-%d")

        # 1) NDVI: try Earth Engine; fallback to synthetic
        ndvi_df = pd.DataFrame()
        if use_ee_if_possible and ee_available:
            ndvi_df = get_ndvi_timeseries_ee(lat, lon, start_date, end_date, buffer_m=buffer_m)
        if ndvi_df is None or ndvi_df.empty:
            ndvi_df = generate_synthetic_ndvi(start_date, end_date)

        # Keep daily cadence - if NDVI timestamps are sparse, forward-fill/interpolate to daily
        ndvi_df = ndvi_df.set_index('date').resample('D').mean().interpolate().reset_index()
        ndvi_df['date'] = pd.to_datetime(ndvi_df['date'])

        # 2) Weather (NASA POWER)
        weather_df = fetch_nasa_power_daily(lat, lon, start_date, end_date)
        if weather_df.empty:
            # fallback synthetic weather
            dates = ndvi_df['date']
            weather_df = pd.DataFrame({
                'date': dates,
                'T2M': np.random.normal(28, 3, size=len(dates)),
                'PRECTOT': np.random.uniform(0, 10, size=len(dates)),
                'RH2M': np.random.uniform(60, 90, size=len(dates))
            })
        # unify column names: T2M -> temp, PRECTOT -> rainfall, RH2M -> humidity
        if 'T2M' in weather_df.columns:
            weather_df = weather_df.rename(columns={'T2M':'temp','PRECTOT':'rainfall','RH2M':'humidity'})
        else:
            # generic fallback names
            if 'T2M_MAX' in weather_df.columns:
                weather_df = weather_df.rename(columns={'T2M':'temp','PRECTOT':'rainfall','RH2M':'humidity'})

        # align by date
        merged = pd.merge(ndvi_df, weather_df, on='date', how='outer').sort_values('date').reset_index(drop=True)
        # clip to sowing->end
        merged = merged[(merged['date'] >= pd.to_datetime(start_date)) & (merged['date'] <= pd.to_datetime(end_date))]
        merged = merged.fillna(method='ffill').fillna(method='bfill').reset_index(drop=True)

        # Add metadata
        merged['crop'] = crop
        merged['district_point'] = name
        merged['lat'] = lat
        merged['lon'] = lon
        merged['sowing_date'] = pd.to_datetime(start_date)
        merged['days_since_sowing'] = (merged['date'] - merged['sowing_date']).dt.days

        # Labels
        merged['stage_label'] = merged['days_since_sowing'].apply(lambda x: label_stage_by_days(x, dur))

        # Save CSV
        fname = f"{OUTPUT_DIR}/{crop}_{name}_{start_date}_dur{dur}.csv".replace(" ","_")
        merged.to_csv(fname, index=False)
        summary_rows.append({
            "crop": crop, "district_point": name, "sowing_date": start_date,
            "rows": len(merged), "csv": fname
        })

print("\nDone. Generated files:")
summary_df = pd.DataFrame(summary_rows)
display(summary_df.head(50))

print(f"\nCSV files are in the folder: {os.path.abspath(OUTPUT_DIR)}")
